### Instruksi Pengerjaan

Buat notebook baru **`Tugas7_[NPM]_[Nama Lengkap].ipynb`**, lalu bangun pipeline analisis lengkap:

---

**A. EXTRACT** *(bobot 10%)*

Baca ketiga sumber data **dari HDFS** (bukan disk lokal) menjadi tiga Spark DataFrame.

**B. TRANSFORM — Join & Kolom Turunan** *(bobot 20%)*

Gabungkan ketiga tabel, tambahkan kolom `total_penjualan` (`qty x harga`).

**C. TRANSFORM — Window Function** *(bobot 20%)*

Tentukan **top-2 menu terlaris** (berdasarkan `total_penjualan`) di **setiap cabang**, gunakan `row_number()`.

**D. TRANSFORM — Spark SQL** *(bobot 20%)*

Menggunakan **Spark SQL murni**, tampilkan total penjualan per `kepala_barista`, diurutkan dari tertinggi.

**E. LOAD** *(bobot 15%)*

Simpan data gabungan (hasil bagian B) ke HDFS sebagai Parquet, path `/user/[username]/tugas7/processed/laporan`, dipartisi berdasarkan `kategori_menu`. Verifikasi dengan membaca kembali dan menghitung jumlah baris.

**F. Ringkasan Eksekutif & Rekomendasi** *(bobot 15%)*

Buat tabel ringkasan (per cabang: total penjualan, jumlah transaksi, rata-rata nilai transaksi), lalu tulis pada markdown cell (**minimal 120 kata**): cabang mana yang berkinerja terbaik, menu apa yang sebaiknya dipromosikan lebih gencar di cabang dengan kinerja terlemah (berdasarkan temuan bagian C), dan alasannya.

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas7_[NPM]_[Nama Lengkap].ipynb` melalui Asprak, paling lambat **sebelum Pertemuan UTS dimulai**.
- Pastikan Hadoop aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Extract | Ketiga sumber berhasil dibaca dari HDFS | 10% |
| B. Join & Transformasi | Join benar, kolom `total_penjualan` tepat | 20% |
| C. Window Function | Top-2 menu per cabang benar menggunakan `row_number()` | 20% |
| D. Spark SQL | Kueri SQL murni berjalan benar sesuai instruksi | 20% |
| E. Load | Data tersimpan ke HDFS sebagai Parquet terpartisi & terverifikasi | 15% |
| F. Ringkasan & Rekomendasi | Insight berbasis data, rekomendasi logis dan relevan | 15% |


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder \
    .appName("Tugas7-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("SparkSession siap. Versi Spark:", spark.version)

26/10/11 13:27:25 WARN Utils: Your hostname, haris resolves to a loopback address: 127.0.1.1; using 192.168.1.57 instead (on interface wlp1s0)
26/10/11 13:27:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/11 13:27:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [5]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/haris/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/haris/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/haris/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/haris/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


## A. EXTRACT: Membaca dari HDFS

In [6]:
BASE = "hdfs://localhost:9000/user/haris/tugas7/raw"

df_transaksi = spark.read.csv(f"{BASE}/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json(f"{BASE}/tugas7_cabang.json")
df_menu = spark.read.csv(f"{BASE}/tugas7_menu.csv", header=True, inferSchema=True)

print("Transaksi:", df_transaksi.count(), "| Cabang:", df_cabang.count(), "| Menu:", df_menu.count())
df_transaksi.printSchema()

Transaksi: 4000 | Cabang: 3 | Menu: 15
root
 |-- trx_id: string (nullable = true)
 |-- cabang_id: integer (nullable = true)
 |-- menu_id: integer (nullable = true)
 |-- qty: integer (nullable = true)



## B. TRANSFORM: Join dan Kolom Turunan

In [7]:
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                          .join(df_menu, on="menu_id", how="inner")

df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))

print("Jumlah baris setelah join:", df_gabungan.count())
df_gabungan.show(5)

Jumlah baris setelah join: 4000
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makana

## C. TRANSFORM: Window Function, Top-2 Menu per Cabang

In [8]:
ringkasan_menu_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu", "kategori_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

top2_menu = ringkasan_menu_cabang \
    .withColumn("peringkat", row_number().over(window_cabang)) \
    .filter(col("peringkat") <= 2)

top2_menu.orderBy("nama_cabang", "peringkat").show(truncate=False)

+-------------------------+---------+--------------+---------------+---------+
|nama_cabang              |nama_menu|kategori_menu |total_penjualan|peringkat|
+-------------------------+---------+--------------+---------------+---------+
|Kopi Nusantara Magelang  |Menu-1   |Makanan Ringan|8505000        |1        |
|Kopi Nusantara Magelang  |Menu-12  |Makanan Ringan|8085000        |2        |
|Kopi Nusantara Semarang  |Menu-13  |Makanan Ringan|8750000        |1        |
|Kopi Nusantara Semarang  |Menu-9   |Makanan Ringan|8715000        |2        |
|Kopi Nusantara Yogyakarta|Menu-12  |Makanan Ringan|9275000        |1        |
|Kopi Nusantara Yogyakarta|Menu-15  |Non-Kopi      |8160000        |2        |
+-------------------------+---------+--------------+---------------+---------+



## D. TRANSFORM: Spark SQL Murni, Penjualan per Kepala Barista

In [9]:
df_gabungan.createOrReplaceTempView("gabungan")

penjualan_barista = spark.sql("""
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM gabungan
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""")
penjualan_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



## E. LOAD: Simpan ke HDFS sebagai Parquet Terpartisi

In [10]:
PATH_OUT = "hdfs://localhost:9000/user/haris/tugas7/processed/laporan"

!hdfs dfs -mkdir -p /user/haris/tugas7/processed

df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(PATH_OUT)

!hdfs dfs -ls /user/haris/tugas7/processed/laporan

Found 4 items
-rw-r--r--   3 haris supergroup          0 2026-10-11 13:33 /user/haris/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - haris supergroup          0 2026-10-11 13:33 /user/haris/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - haris supergroup          0 2026-10-11 13:33 /user/haris/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - haris supergroup          0 2026-10-11 13:33 /user/haris/tugas7/processed/laporan/kategori_menu=Non-Kopi


## F. Ringkasan Eksekutif dan Rekomendasi

In [13]:
PATH_OUT = "hdfs://localhost:9000/user/haris/tugas7/processed/laporan"
df_verifikasi = spark.read.parquet(PATH_OUT)

ringkasan_cabang = df_verifikasi.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

ringkasan_cabang.show(truncate=False)

+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|nama_cabang              |kota      |kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Semarang  |Semarang  |Nadia         |90542000       |1393            |64997.8463747308         |
|Kopi Nusantara Magelang  |Magelang  |Nadia         |87138000       |1302            |66926.26728110599        |
|Kopi Nusantara Yogyakarta|Yogyakarta|Reza          |84645000       |1305            |64862.06896551724        |
+-------------------------+----------+--------------+---------------+----------------+-------------------------+



Cabang terbaik. Kopi Nusantara Semarang berkinerja terbaik dengan total penjualan Rp90.542.000 dari 1.393 transaksi, tertinggi pada kedua ukuran. Magelang berada di posisi kedua (Rp87.138.000; 1.302 transaksi) dan memiliki rata-rata nilai transaksi tertinggi, Rp66.926, sedangkan Semarang Rp64.998. Yogyakarta terlemah dengan Rp84.645.000, selisih sekitar Rp5,9 juta (sekitar 7%) dari Semarang.

Penyebab kelemahan Yogyakarta. Jumlah transaksinya (1.305) hampir sama dengan Magelang (1.302), tetapi rata-rata nilai transaksinya lebih rendah (Rp64.862 berbanding Rp66.926). Jadi masalahnya bukan sepinya pengunjung, melainkan nilai belanja per transaksi.

Menu yang dipromosikan di Yogyakarta. Dua menu terlaris di sana adalah Menu-12 (Rp9.275.000) dan Menu-15 (Rp8.160.000), sehingga promosi pada keduanya paling mungkin menggerakkan omzet. Menu-12 (Makanan Ringan, Rp35.000) juga masuk tiga besar di Magelang dan Semarang, jadi terbukti laku lintas cabang dan berharga tinggi. Saya sarankan paket bundling Menu-12 dan Menu-15 dengan minuman kopi untuk menaikkan nilai belanja per transaksi, serta mengenalkan Menu-9 (Rp35.000) yang kuat di Semarang dan sudah berada di peringkat ketiga Yogyakarta (Rp7.175.000).

In [ ]:
spark.stop()
print('SparkSession ditutup.')